# S1: Water Quality + Geography + Soil (Daily) Merge

Secondary merge per `MERGE.md` (§3, **S1**). Attaches each station's static
geography (HUC-12/10/8, derived `county_fips`) and soil (`mukey` + SSURGO
attributes) to every water-quality measurement event, alongside the daily
climate/streamflow already assembled in P1. Both inputs are `03a` primary
outputs — nothing here reaches back to `02_clean`.

**Inputs** (both `data/03a_merge_primary/...`):
- `wq-daily-environment.csv` (**P1**) — one row per WQ measurement event
  (`MonitoringLocationIdentifier` + `ActivityStartDateTime`), with daily
  PRISM/ISU climate and USGS streamflow. This is the base table; its grain is
  preserved.
- `station-geo-soil.csv` (**P2**) — one row per station, carrying HUC-12/10/8,
  `county_fips`, `mukey`, and the map unit's soil attributes.

**Join:** left join on `MonitoringLocationIdentifier`. P2 is one row per
station, so it broadcasts each station's static context onto all of that
station's measurement events without changing the grain.

**Overlapping columns.** P1 already carries the station-identity columns
(`OrganizationIdentifier`, `MonitoringLocationName`, `MonitoringLocationTypeName`,
`HUCEightDigitCode`, lat/lon, `StateCode`, `CountyCode`, `ProviderName`) because
P1 joined the station table in too. Those are dropped from the P2 side before
merging — keeping P1's copies — so S1 adds only the *new* geo/soil columns and
no `_x`/`_y` suffixes appear.

**Output:** `data/03b_merge_secondary/wq-geo-soil-daily.csv`, one row per WQ
measurement event.

> Soil coverage is ~65% by design (P2 note): stations sit in/next to water
> map units that carry no soil component, so `mukey`/soil attributes are null
> for the rest — an expected property of station placement, not a join defect.

In [1]:
import os

import pandas as pd

PRIMARY = "../../data/03a_merge_primary"
OUT_DIR = "../../data/03b_merge_secondary"
OUT_FILE = f"{OUT_DIR}/wq-geo-soil-daily.csv"

KEY = "MonitoringLocationIdentifier"
GRAIN = [KEY, "ActivityStartDateTime"]

# HUC / FIPS / mukey codes must stay strings so leading zeros survive the round-trip.
CODE_DTYPES = {
    "HUCEightDigitCode": str,
    "county_fips": str,
    "huc12_code": str,
    "huc10_code": str,
    "huc8_code": str,
    "mukey": str,
}

## Step 1: Load P1 (measurement + daily environment)

P1 is the base grain — one row per `(station, timestamp)`. Assert that grain is
intact before joining anything onto it.

In [2]:
df = pd.read_csv(
    f"{PRIMARY}/wq-daily-environment.csv",
    parse_dates=["ActivityStartDateTime"],
    dtype={k: v for k, v in CODE_DTYPES.items() if k == "HUCEightDigitCode"},
    low_memory=False,
)
print(f"P1 wq-daily-environment: {df.shape}")
assert not df.duplicated(subset=GRAIN).any(), "P1 grain violated: duplicate (station, timestamp) rows"
n_rows_in = len(df)

P1 wq-daily-environment: (48251, 84)


## Step 2: Load P2 (station geography + soil), drop columns P1 already has

P2 is one row per station. The station-identity columns it shares with P1 are
dropped (except the join key) so the merge contributes only the new geo/soil
context and produces no duplicate/suffixed columns.

In [3]:
df_geo = pd.read_csv(
    f"{PRIMARY}/station-geo-soil.csv",
    dtype={k: v for k, v in CODE_DTYPES.items() if k != "HUCEightDigitCode"},
)
print(f"P2 station-geo-soil: {df_geo.shape}")
assert not df_geo.duplicated(subset=[KEY]).any(), "P2 is not 1 row per station"

# Everything P1 already carries (besides the join key) is dropped from the P2 side.
overlap = [c for c in df_geo.columns if c in df.columns and c != KEY]
df_geo = df_geo.drop(columns=overlap)
new_cols = [c for c in df_geo.columns if c != KEY]
print(f"Dropped {len(overlap)} overlapping columns kept from P1: {overlap}")
print(f"Adding {len(new_cols)} new geo/soil columns: {new_cols}")

P2 station-geo-soil: (1666, 28)
Dropped 9 overlapping columns kept from P1: ['OrganizationIdentifier', 'MonitoringLocationName', 'MonitoringLocationTypeName', 'HUCEightDigitCode', 'LatitudeMeasure', 'LongitudeMeasure', 'StateCode', 'CountyCode', 'ProviderName']
Adding 18 new geo/soil columns: ['county_fips', 'huc12_code', 'huc12_name', 'huc10_code', 'huc8_code', 'huc12_acres', 'mukey', 'map_unit_symbol', 'survey_area', 'soil_match_method', 'soil_match_distance_m', 'map_unit_name', 'dominant_component', 'dominant_component_pct', 'hydrologic_group', 'drainage_class', 'ksat_mean', 'awc_mean']


## Step 3: Merge P2 onto P1 (left join on station id)

In [4]:
df = df.merge(df_geo, on=KEY, how="left")

print(f"Merged shape: {df.shape}")
print(f"Station geo/soil matched: {df['huc12_code'].notna().mean():.1%} of measurement rows")
assert len(df) == n_rows_in, "P2 join fanned out measurement rows"
df.head(3)

Merged shape: (48251, 102)
Station geo/soil matched: 100.0% of measurement rows


,MonitoringLocationIdentifier,ActivityStartDateTime,"Temperature, water_value","Temperature, water_unit",Dissolved oxygen (DO)_value,Dissolved oxygen (DO)_unit,pH_value,pH_unit,Nitrate_value,Nitrate_unit,...,survey_area,soil_match_method,soil_match_distance_m,map_unit_name,dominant_component,dominant_component_pct,hydrologic_group,drainage_class,ksat_mean,awc_mean
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2017-07-18 15:00:00,19.820,deg C,8.75,mg/L,8.330,std units,NaN,NaN,...,IA005,within,0.0,"Ion silt loam, 0 to 2 percent slopes",Ion,95.0,B,Moderately well drained,9.00,0.213
1,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2023-07-19 11:00:00,16.372,deg C,12.24,mg/L,8.215,std units,NaN,NaN,...,IA005,within,0.0,"Ion silt loam, 0 to 2 percent slopes",Ion,95.0,B,Moderately well drained,9.00,0.213
2,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2017-07-17 17:00:00,19.920,deg C,7.62,mg/L,8.195,std units,NaN,NaN,...,IA031,within,0.0,"Colo silt loam, 0 to 2 percent slopes, occasio...",Colo,90.0,C/D,Poorly drained,4.33,0.181


## Step 4: Final checks and save

In [5]:
print(f"Final shape: {df.shape}")
print(f"Final columns: {list(df.columns)}")
assert not df.duplicated(subset=GRAIN).any(), "Output grain violated: duplicate (station, timestamp) rows"
assert len(df) == n_rows_in, "Row count changed vs. P1 base"

print("\nCoverage (share of measurement rows):")
print(f"  HUC-12 watershed:  {df['huc12_code'].notna().mean():.1%}")
print(f"  county_fips:       {df['county_fips'].notna().mean():.1%}")
print(f"  Soil map unit:     {df['mukey'].notna().mean():.1%}")
print(f"  Soil attributes:   {df['hydrologic_group'].notna().mean():.1%}")

Final shape: (48251, 102)
Final columns: ['MonitoringLocationIdentifier', 'ActivityStartDateTime', 'Temperature, water_value', 'Temperature, water_unit', 'Dissolved oxygen (DO)_value', 'Dissolved oxygen (DO)_unit', 'pH_value', 'pH_unit', 'Nitrate_value', 'Nitrate_unit', 'Nitrite_value', 'Nitrite_unit', 'Nitrate + Nitrite_value', 'Nitrate + Nitrite_unit', 'Ammonia-nitrogen_value', 'Ammonia-nitrogen_unit', 'Kjeldahl nitrogen_value', 'Kjeldahl nitrogen_unit', 'Orthophosphate_value', 'Orthophosphate_unit', 'Phosphate-phosphorus_value', 'Phosphate-phosphorus_unit', 'Total Phosphorus, mixed forms_value', 'Total Phosphorus, mixed forms_unit', 'Chloride_value', 'Chloride_unit', 'Sulfate_value', 'Sulfate_unit', 'Specific conductance_value', 'Specific conductance_unit', 'Total dissolved solids_value', 'Total dissolved solids_unit', 'Total suspended solids_value', 'Total suspended solids_unit', 'Turbidity_value', 'Turbidity_unit', 'Escherichia coli_value', 'Escherichia coli_unit', 'Chlorophyll a,

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 48,251 rows x 102 cols -> ../../data/03b_merge_secondary/wq-geo-soil-daily.csv
